In [1]:
import torch
import numpy as numpy
import os
import shutil
import pandas as pd
from torchvision import models
import gc

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
torch.cuda.empty_cache()

cuda


In [2]:
import torch
import sys
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

/home/undergrad/2026/ojonesma/Spring2026/EyeClassifierEnv/bin/python
2.10.0+cu126
12.6
True


## Efficient Net
#### Olivia Jones-Martin

In [3]:
from utilities import *

# setting path variables for later use
TRAINING_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/a. Training Set"
VALIDATION_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/b. Validation Set"
TESTING_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/c. Testing Set"

GROUND_TRUTHS_PATH = "./A. RFMiD_All_Classes_Dataset/3. GroundtruthsChallenge"
'''
GROUND_TRUTHS_PATH can also point to the folder containing ground truths for all classes
The way my dataset is set up is that there's 3 folders, one for the images, one for the metadata for all classes,
and one for the metadata in which any disease with less than 10 instances get grouped into the label other
This can be modified depending on which set of metadata you're using
-Olivia Jones-Martin
'''

TRAINING_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "a. RFMiD_Training_Labels.csv")
VALIDATION_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "b. RFMiD_Validation_Labels.csv")
TEST_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "c. RFMiD_Testing_Labels.csv")

Loss function being used is [binary cross entropy with logits loss](https://docs.pytorch.org/docs/1.7.1/generated/torch.nn.BCEWithLogitsLoss.html#torch.nn.BCEWithLogitsLoss). Weights are calculated for the pos_weight parameter for the function.

In [4]:
# Get counts for each of the 
trainingMetadata = pd.read_csv(TRAINING_METADATA_PATH)
diseaseLabels = trainingMetadata.columns[1:]

# counts includes every class except for ID
counts = trainingMetadata.iloc[:, 1:].sum()
print("Disease Counts:")
print(counts)

# total amount of images and labels in the training set
image_count = trainingMetadata.shape[0]
label_count = counts.shape[0]
print(f"Image count: {image_count}")
print(f"Label count: {label_count}")

'''
If there is an instance where a disease appears 0 times, give a warning and
set the count value to one to prevent a divide by zero error when calculating
the weights
'''
for i in range(label_count):
    if counts.iloc[i] == 0:
        counts.iloc[i] = 1
        print(f"WARNING! Zero instances of {diseaseLabels[i]} in dataset")

'''
Calculate pos_weights for binary cross entropy with logits loss
pos weight for a class should be (negative counts of class)/(positive counts of class)
Documentation in reference 2
'''
pos_weights = [(image_count - count)/count for count in counts]
print("pos_weigths for BCEWithLogitsLoss")
print(pos_weights)

Disease Counts:
Disease_Risk    1519
DR               376
ARMD             100
MH               317
DN               138
MYA              101
BRVO              73
TSLN             186
ERM               14
LS                47
MS                15
CSR               37
ODC              282
CRVO              28
TV                 6
AH                16
ODP               65
ODE               58
ST                 5
AION              17
PT                11
RT                14
RS                43
CRS               32
EDN               15
RPEC              22
MHL               11
RP                 6
OTHER             34
dtype: int64
Image count: 1920
Label count: 29
pos_weigths for BCEWithLogitsLoss
[0.26398946675444374, 4.1063829787234045, 18.2, 5.056782334384858, 12.91304347826087, 18.00990099009901, 25.301369863013697, 9.32258064516129, 136.14285714285714, 39.851063829787236, 127.0, 50.891891891891895, 5.808510638297872, 67.57142857142857, 319.0, 119.0, 28.53846153846154, 32.1034482758

Load the images

In [5]:
from torch.utils.data import DataLoader, Subset

# metadata for training dataset already loaded in previous parts, only need to load in validatoin and
# test metadata dataframes
validationMetadata = pd.read_csv(VALIDATION_METADATA_PATH)
testMetadata = pd.read_csv(TEST_METADATA_PATH)

# Number of modules used for bagging (bootstrap aggregating)
NUM_OF_MODULES = 5
BATCH_SIZE = 6
NUM_OF_WORKERS = 0

training_set_size = trainingMetadata.shape[0]
validation_set_size = validationMetadata.shape[0]
test_set_size = testMetadata.shape[0]

'''
efficientnet_b1 uses images sizes of 224x224 (reference 3),
normilization used is RGB means of [0.485, 0.456, 0.406] and
standard deviations of [0.229, 0.224, 0.225]. means and std
deviation are given by reference 1 in section 5.2
'''
augmenter = dataAugmenter(image_size = (224, 224),
                          norm_mean=(0.485, 0.456, 0.406),
                          norm_std=(0.229, 0.224, 0.225),
                          useCutOut=True)
# Augmenter class used to provide transformations

training_dataset = FundusImageDataset(metadata=trainingMetadata,
                                      image_directory=TRAINING_SET_PATH,
                                      transform=augmenter.transform_train)
validation_dataset = FundusImageDataset(metadata=validationMetadata,
                                        image_directory=VALIDATION_SET_PATH,
                                        transform=augmenter.transform_test)
test_dataset = FundusImageDataset(metadata=testMetadata,
                                  image_directory=TESTING_SET_PATH,
                                  transform=augmenter.transform_test)

'''
Dataloader takes a "Map-style dataset" that has the __getitem__() and __len__() protocols
to access data
this class is implemented in utilities.py
multiple random subsets of training and validation sets are picked out for bootstrap sampling, testing_loader can be created here though
'''
testing_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Model Setup
setting up the use of efficientNet v3, using ensamble/bagging

In [6]:
def memorySummary():
    print("Current GPU Memory Usage:")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.2f} GB")
    print(f"\nTotal GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Training the model

In [7]:
from torchmetrics.classification import MultilabelF1Score

# Create loss function with given weights calculated
pos_weights = torch.FloatTensor(pos_weights).to(device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# array used to average out predictions
total_predictions = np.zeros((test_set_size, label_count))

# targets for using F1 test
targets = testMetadata.iloc[:, 1:].to_numpy()
targets = torch.from_numpy(targets)

'''
Previous attempt used several modules in a nn.ModuleList datatype but this used too much memory in the server so only one module
is evaluated at a time, cuda cache is also emptied to prevent any issues
'''
gc.collect()
torch.cuda.empty_cache()
for i in range(NUM_OF_MODULES):
    print(f"Module Iteration: {i + 1}")
    model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)

    # replace feature layer with desired amount of output classes
    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(in_features, label_count)
    model = model.to(device)

    # set up the optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

    # Create a dataloader for subset of training and validation samples for training, each module uses a different subset
    indices = np.random.choice(a=training_set_size,
                               size=training_set_size,
                               replace=True)
    
    subset = Subset(training_dataset, indices)
    training_loader = DataLoader(dataset=subset,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_OF_WORKERS,
                                 pin_memory=False)
    
    # create validation loader
    indices = np.random.choice(a=validation_set_size,
                               size=validation_set_size,
                               replace=True)
    
    subset = Subset(validation_dataset, indices)
    validation_loader = DataLoader(dataset=subset,
                                   batch_size=BATCH_SIZE,
                                   shuffle=True,
                                   num_workers=NUM_OF_WORKERS)
    
    # Create evaluator for this module
    evaluator = ModelEvaluator(training_loader=training_loader,
                               validation_loader=validation_loader,
                               testing_loader=testing_loader,
                               loss_criterion=criterion,
                               optimizer=optimizer,
                               device=device)

    # Train model
    print(f"Memory before training module {i+1}:")
    memorySummary()
    
    evaluator.train(model=model, epoch_count=15, verbose=True)
    predictions = evaluator.testProbabilities(model=model)

    # get individual F1 score for sub model
    total_predictions += predictions
    predictions = (torch.from_numpy(predictions) > 0.5)
    f1 = MultilabelF1Score(num_labels=label_count)
    score = f1(predictions, targets)
    print(f"Model {i+1} F1 score:")
    print(score)

    # get confusion matrices
    matrices = evaluator.test(model)
    print(f"Model {i+1} confusion matrices")
    print(matrices)

    # Clear cuda memory immediately after training
    del model, optimizer, evaluator, training_loader, validation_loader, subset, indices
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Memory after clearing module {i+1}:")
    memorySummary()

average_predictions = total_predictions / NUM_OF_MODULES
average_predictions = (torch.from_numpy(average_predictions) > 0.5).float()
print("Average predictions")
print(average_predictions)

Module Iteration: 1
Memory before training module 1:
Current GPU Memory Usage:
Allocated: 0.03 GB
Reserved: 0.04 GB
Free: 50.86 GB

Total GPU Memory: 50.90 GB
Epoch: 1
Loss: 1.2704636252827226 Accuracy: 0.6466594827586207
Validation Loss: 1.4403884038291108 Val Accuracy: 0.6526400862068965
Epoch: 2
Loss: 1.138215591767787 Accuracy: 0.6795797413793103
Validation Loss: 1.691861747548934 Val Accuracy: 0.6991918103448276
Epoch: 3
Loss: 0.9025502276395343 Accuracy: 0.759698275862069
Validation Loss: 2.091085219695195 Val Accuracy: 0.7742456896551724
Epoch: 4
Loss: 0.7119217403430185 Accuracy: 0.8221264367816092
Validation Loss: 2.62644742832312 Val Accuracy: 0.8253771551724138
Epoch: 5
Loss: 0.5994259074883485 Accuracy: 0.8497485632183908
Validation Loss: 3.2907760553445558 Val Accuracy: 0.8661099137931034
Epoch: 6
Loss: 0.4951913174834976 Accuracy: 0.8745869252873564
Validation Loss: 4.986048176107354 Val Accuracy: 0.8983297413793103
Epoch: 7
Loss: 0.4736315167332501 Accuracy: 0.8804777298

In [ ]:

def train_fold(int):
    return

## Accuracy using F1 Test

In [8]:
f1 = MultilabelF1Score(num_labels=label_count)
score = f1(average_predictions, targets)

print(average_predictions)

print("Score")
print(score)

# try oversampling rare class or class weighted sampling
# improve augmentations
# use full validation set instead of sampling subsets
# unfreeze rest of weights for efficientnet
# use recall curve

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 1.,  ..., 0., 0., 0.]])
Score
tensor(0.3549)
